# 05 - Validar RiverSP Top 1 do ranking por proximidade

Este notebook baixa e valida apenas o novo candidato leve `RIVERSP` apontado como rank 1 pelo notebook 04. A validação é feita pela geometria interna do produto, não pela bounding box do metadado.

## Contexto

O notebook 04 organizou 1166 metadados SWOT, mas todos os candidatos ficaram limitados a `bounding_box`. Isso é útil para filtrar cobertura potencial, porém não confirma suporte observacional real: uma bounding box pode cobrir os 13 exutórios mesmo quando as feições internas do produto estão longe dos pontos.

O RiverSP anterior, `cycle 055`, `pass 255`, `tile SA`, foi rejeitado no notebook 03 porque as feições internas ficaram a aproximadamente 161,7 km a 163,7 km dos exutórios.

## Objetivo

Validar o candidato `RIVERSP cycle 055 pass 227 tile SA` baixando somente o ZIP desse produto, abrindo a camada espacial interna, medindo as distâncias reais até os 13 exutórios e classificando o suporte preliminar.

In [ ]:
from __future__ import annotations

import json
import logging
import tempfile
import zipfile
from pathlib import Path
from urllib.parse import urlparse

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import LineString


In [ ]:
def find_project_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / 'requirements.txt').exists() and (candidate / 'src' / 'check_environment.py').exists():
            return candidate
    fallback = Path.home() / 'mystorage' / 'PPGGAG1889' / 'atividade3_swot'
    if fallback.exists():
        return fallback
    raise RuntimeError('FALHA: rode este notebook dentro do repositorio atividade3_swot.')

PROJECT_ROOT = find_project_root(Path.cwd())
EXUTORIOS_CSV = PROJECT_ROOT / 'dados' / 'exutorios.csv'
RANKING_CSV = PROJECT_ROOT / 'outputs' / 'tabelas' / 'ranking_candidatos_swot_proximidade.csv'
VALIDACAO_ANTERIOR_CSV = PROJECT_ROOT / 'outputs' / 'tabelas' / 'validacao_riversp_exutorios.csv'
RAW_DIR = PROJECT_ROOT / 'dados' / 'raw' / 'swot' / 'riversp'
INTERMEDIATE_DIR = PROJECT_ROOT / 'dados' / 'intermediarios' / 'swot' / 'riversp'
OUTPUT_TABLE = PROJECT_ROOT / 'outputs' / 'tabelas' / 'validacao_riversp_top1_ranking_exutorios.csv'
OUTPUT_FIGURE = PROJECT_ROOT / 'outputs' / 'figuras' / '05_validar_riversp_top1_ranking.png'
LOG_FILE = PROJECT_ROOT / 'outputs' / 'logs' / '05_validar_riversp_top1_ranking.log'

for path in [RAW_DIR, INTERMEDIATE_DIR, OUTPUT_TABLE.parent, OUTPUT_FIGURE.parent, LOG_FILE.parent]:
    path.mkdir(parents=True, exist_ok=True)

logging.basicConfig(filename=LOG_FILE, filemode='w', level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
print('OK raiz do projeto:', PROJECT_ROOT)
print('OK raw:', RAW_DIR)
print('OK log:', LOG_FILE)


## Entradas

A célula abaixo lê os exutórios, o ranking do notebook 04 e, quando existir, a validação anterior do RiverSP rejeitado. As coordenadas não são alteradas.

In [ ]:
expected_columns = ['id', 'latitude', 'longitude']
if not EXUTORIOS_CSV.exists():
    raise FileNotFoundError(f'FALHA: arquivo de exutorios nao encontrado: {EXUTORIOS_CSV}')
if not RANKING_CSV.exists():
    raise FileNotFoundError(f'FALHA: ranking do notebook 04 nao encontrado: {RANKING_CSV}')

exutorios = pd.read_csv(EXUTORIOS_CSV)
if list(exutorios.columns) != expected_columns:
    raise ValueError(f'FALHA: colunas esperadas {expected_columns}, colunas encontradas {list(exutorios.columns)}')
if len(exutorios) != 13:
    raise ValueError(f'FALHA: esperados 13 exutorios, encontrados {len(exutorios)}')
exutorios['latitude'] = pd.to_numeric(exutorios['latitude'], errors='raise')
exutorios['longitude'] = pd.to_numeric(exutorios['longitude'], errors='raise')

ranking = pd.read_csv(RANKING_CSV, dtype={'cycle': str, 'pass': str, 'tile': str})
validacao_anterior = pd.read_csv(VALIDACAO_ANTERIOR_CSV) if VALIDACAO_ANTERIOR_CSV.exists() else pd.DataFrame()

print('OK exutorios:', len(exutorios))
print('OK ranking:', len(ranking))
print('OK validacao anterior:', 'sim' if not validacao_anterior.empty else 'nao encontrada')
display(exutorios)


## Seleção controlada do candidato

O candidato selecionado deve ser exatamente o rank 1 `RIVERSP`, `cycle 055`, `pass 227`, `tile SA`, com o `granule_id` esperado. Se o ranking mudar, o notebook falha de forma explícita para evitar baixar outro produto por engano.

In [ ]:
EXPECTED = {
    'rank': 1,
    'produto': 'RIVERSP',
    'cycle': '055',
    'pass': '227',
    'tile': 'SA',
    'granule_id': 'SWOT_L2_HR_RiverSP_Reach_055_227_SA_20260829T001130_20260829T002036_PID0_01',
}

rank1 = ranking[ranking['rank'].astype(int).eq(1)]
if rank1.empty:
    raise ValueError('FALHA: rank 1 nao encontrado no ranking do notebook 04.')

candidate = rank1.iloc[0].to_dict()
for key, expected in EXPECTED.items():
    actual = candidate.get(key)
    if key in ['cycle', 'pass']:
        actual = str(actual).zfill(3)
    elif key == 'rank':
        actual = int(actual)
    else:
        actual = str(actual)
    if actual != expected:
        raise ValueError(f'FALHA: candidato rank 1 inesperado para {key}: esperado {expected}, encontrado {actual}')

url = str(candidate['download_url'])
granule_id = str(candidate['granule_id'])
expected_mb = float(candidate['tamanho_mb']) if pd.notna(candidate.get('tamanho_mb')) else None
filename = Path(urlparse(url).path).name or f'{granule_id}.zip'
if not filename.lower().endswith('.zip'):
    raise ValueError(f'FALHA: URL do candidato RIVERSP nao aponta para .zip: {url}')
zip_path = RAW_DIR / filename

logging.info('Candidato RIVERSP top 1 selecionado: %s', json.dumps(candidate, ensure_ascii=False))
print('OK candidato RIVERSP top 1 validado')
print('granule_id:', granule_id)
print('url:', url)
print('tamanho esperado MB:', expected_mb)
print('destino:', zip_path)


## Download controlado

A próxima célula baixa somente este ZIP RiverSP, se ele ainda não existir em `dados/raw/swot/riversp/`. Nenhum PIXC é baixado neste notebook.

In [ ]:
try:
    import earthaccess
except Exception as exc:
    logging.exception('Falha ao importar earthaccess')
    raise RuntimeError('FALHA: earthaccess nao esta instalado. Rode pip install -r requirements.txt.') from exc

already_exists = zip_path.exists() and zip_path.stat().st_size > 0
if already_exists:
    downloaded_paths = [zip_path]
    status_download = 'arquivo_ja_existia'
else:
    try:
        earthaccess.login(strategy='interactive', persist=True)
        downloaded_paths = earthaccess.download(url, local_path=RAW_DIR, threads=1, show_progress=True)
        status_download = 'baixado'
    except Exception as exc:
        logging.exception('Falha no download RiverSP top 1: %s', url)
        raise RuntimeError('FALHA: download do RIVERSP top 1 nao concluido. Verifique login Earthdata, URL e conectividade.') from exc

if not downloaded_paths:
    raise RuntimeError('FALHA: earthaccess.download nao retornou caminho baixado.')
zip_path = Path(downloaded_paths[0])
if not zip_path.exists():
    candidate_path = RAW_DIR / filename
    if candidate_path.exists():
        zip_path = candidate_path
    else:
        raise FileNotFoundError(f'FALHA: arquivo baixado nao encontrado: {downloaded_paths[0]}')

downloaded_mb = zip_path.stat().st_size / (1024 * 1024)
logging.info('granule_id: %s', granule_id)
logging.info('URL usada: %s', url)
logging.info('Arquivo: %s', filename)
logging.info('Tamanho esperado MB: %s', expected_mb)
logging.info('Tamanho local MB: %.3f', downloaded_mb)
logging.info('Caminho local: %s', zip_path)
logging.info('Status download: %s', status_download)

print('OK download RiverSP top 1:', status_download)
print('arquivo:', zip_path)
print('tamanho local MB:', round(downloaded_mb, 3))


## Inspeção do ZIP e leitura espacial

Antes de assumir o esquema interno do produto, o ZIP é validado, seus arquivos são listados e a camada espacial é identificada. A leitura prioriza shapefile de `reach`, quando existir.

In [ ]:
if not zipfile.is_zipfile(zip_path):
    raise zipfile.BadZipFile(f'FALHA: arquivo nao e um ZIP valido: {zip_path}')

with zipfile.ZipFile(zip_path) as zf:
    members = zf.infolist()
    zip_inventory = pd.DataFrame([
        {'nome': m.filename, 'tamanho_bytes': m.file_size, 'compactado_bytes': m.compress_size}
        for m in members
    ]).sort_values('nome').reset_index(drop=True)

logging.info('Arquivos internos do ZIP: %s', len(zip_inventory))
for name in zip_inventory['nome'].tolist():
    logging.info('ZIP member: %s', name)

vector_ext = ('.shp', '.gpkg', '.geojson', '.json')
table_ext = ('.csv', '.parquet')
vector_members = [n for n in zip_inventory['nome'] if n.lower().endswith(vector_ext)]
table_members = [n for n in zip_inventory['nome'] if n.lower().endswith(table_ext)]

print('OK arquivos no ZIP:', len(zip_inventory))
print('vetoriais candidatos:', vector_members)
print('tabelares candidatos:', table_members)
display(zip_inventory)


In [ ]:
def member_priority(name: str) -> tuple[int, str]:
    lower = name.lower()
    if lower.endswith('.shp') and 'reach' in lower:
        return (0, lower)
    if lower.endswith('.shp'):
        return (1, lower)
    if lower.endswith(('.gpkg', '.geojson')):
        return (2, lower)
    if lower.endswith('.json'):
        return (3, lower)
    return (9, lower)

if not vector_members:
    raise RuntimeError('FALHA: nenhum arquivo vetorial reconhecido dentro do ZIP RiverSP top 1.')

selected_member = sorted(vector_members, key=member_priority)[0]
logging.info('Arquivo vetorial selecionado para leitura: %s', selected_member)
print('OK camada/arquivo selecionado:', selected_member)

read_errors = []
riversp_gdf = None
try:
    riversp_gdf = gpd.read_file(f'zip://{zip_path}!{selected_member}')
except Exception as exc:
    read_errors.append(f'zip:// falhou: {exc}')

if riversp_gdf is None:
    extract_dir = INTERMEDIATE_DIR / zip_path.stem
    extract_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as zf:
        related_prefix = Path(selected_member).with_suffix('').name
        for member in zf.namelist():
            base = Path(member).with_suffix('').name
            if base == related_prefix:
                zf.extract(member, extract_dir)
    extracted_candidates = list(extract_dir.rglob(Path(selected_member).name))
    if not extracted_candidates:
        raise RuntimeError('FALHA: nao foi possivel extrair o arquivo vetorial selecionado.')
    try:
        riversp_gdf = gpd.read_file(extracted_candidates[0])
    except Exception as exc:
        read_errors.append(f'extracao falhou: {exc}')
        logging.error('Erros de leitura: %s', read_errors)
        raise RuntimeError('FALHA: nao foi possivel ler a camada RiverSP top 1 com geopandas.') from exc

if riversp_gdf.empty:
    raise RuntimeError('FALHA: camada RiverSP top 1 lida, mas sem feicoes.')
if riversp_gdf.crs is None:
    riversp_gdf = riversp_gdf.set_crs('EPSG:4326', allow_override=True)

logging.info('CRS RiverSP top 1: %s', riversp_gdf.crs)
logging.info('Total feicoes RiverSP top 1: %s', len(riversp_gdf))
logging.info('Colunas RiverSP top 1: %s', list(riversp_gdf.columns))
print('OK RiverSP top 1 carregado')
print('CRS:', riversp_gdf.crs)
print('feicoes:', len(riversp_gdf))
print('colunas:', list(riversp_gdf.columns))
display(riversp_gdf.head())


## Critério de suporte preliminar

O suporte `sim` é atribuído quando o exutório está a até 500 m da feição RiverSP mais próxima. Esse limiar é conservador para uma validação preliminar ponto-feição: distâncias muito superiores indicam incompatibilidade espacial clara; casos próximos com leitura incompleta ou atributos ambíguos podem ser marcados como `indeterminado`.

In [ ]:
SUPPORT_DISTANCE_M = 500
METRIC_CRS = 'EPSG:32723'

points_gdf = gpd.GeoDataFrame(
    exutorios.copy(),
    geometry=gpd.points_from_xy(exutorios['longitude'], exutorios['latitude']),
    crs='EPSG:4326',
)
points_m = points_gdf.to_crs(METRIC_CRS)
riversp_m = riversp_gdf.to_crs(METRIC_CRS)

id_candidates = ['reach_id', 'node_id', 'river_name', 'name', 'id', 'feature_id', 'lake_id']
id_columns = [c for c in id_candidates if c in riversp_m.columns]
attribute_columns = [c for c in ['reach_id', 'node_id', 'river_name', 'wse', 'width', 'slope', 'quality_f', 'dark_frac', 'geometry'] if c in riversp_m.columns]
if not attribute_columns:
    attribute_columns = list(riversp_m.columns[: min(8, len(riversp_m.columns))])

if not id_columns:
    logging.warning('Nenhuma coluna identificadora padrao encontrada no RiverSP top 1.')

rows = []
nearest_lines = []
for _, point in points_m.iterrows():
    distances = riversp_m.geometry.distance(point.geometry)
    if distances.empty or distances.isna().all():
        rows.append({
            'id': point['id'], 'latitude': point['latitude'], 'longitude': point['longitude'],
            'cycle': EXPECTED['cycle'], 'pass': EXPECTED['pass'], 'tile': EXPECTED['tile'], 'granule_id': granule_id,
            'distancia_m_feicao_riversp': pd.NA, 'feicao_mais_proxima_id': '', 'river_name': '',
            'suporte_riversp': 'indeterminado', 'observacoes': 'Nao foi possivel calcular distancia ate feicoes RiverSP.',
        })
        continue

    nearest_idx = distances.idxmin()
    nearest = riversp_m.loc[nearest_idx]
    dist_m = float(distances.loc[nearest_idx])

    feature_id = ''
    for col in id_columns:
        value = nearest.get(col)
        if pd.notna(value):
            feature_id = str(value)
            break

    river_name = str(nearest.get('river_name')) if 'river_name' in riversp_m.columns and pd.notna(nearest.get('river_name')) else ''
    attrs = {col: nearest.get(col) for col in attribute_columns if col != 'geometry'}

    if not id_columns:
        support = 'indeterminado'
        reason = 'sem coluna identificadora padrao; suporte mantido indeterminado'
    elif dist_m <= SUPPORT_DISTANCE_M:
        support = 'sim'
        reason = f'distancia <= {SUPPORT_DISTANCE_M} m'
    else:
        support = 'nao'
        reason = f'distancia > {SUPPORT_DISTANCE_M} m'

    nearest_point = riversp_m.geometry.loc[nearest_idx].interpolate(riversp_m.geometry.loc[nearest_idx].project(point.geometry))
    nearest_lines.append({'id': point['id'], 'geometry': LineString([point.geometry, nearest_point])})

    rows.append({
        'id': point['id'],
        'latitude': point['latitude'],
        'longitude': point['longitude'],
        'cycle': EXPECTED['cycle'],
        'pass': EXPECTED['pass'],
        'tile': EXPECTED['tile'],
        'granule_id': granule_id,
        'distancia_m_feicao_riversp': round(dist_m, 2),
        'feicao_mais_proxima_id': feature_id,
        'river_name': river_name,
        'suporte_riversp': support,
        'observacoes': f'{reason}; attrs: {attrs}',
    })

validation = pd.DataFrame(rows)
validation.to_csv(OUTPUT_TABLE, index=False, encoding='utf-8')
logging.info('Tabela de validacao RiverSP top 1 salva: %s', OUTPUT_TABLE)
logging.info('Suporte RiverSP top 1: %s', validation['suporte_riversp'].value_counts(dropna=False).to_dict())
print('OK tabela salva:', OUTPUT_TABLE)
print(validation['suporte_riversp'].value_counts(dropna=False))
display(validation)


## Comparação com o RiverSP rejeitado

A comparação abaixo usa a validação anterior quando ela existir. O candidato anterior era `cycle 055`, `pass 255`, `tile SA`, com distâncias reais de aproximadamente 161,7 km a 163,7 km. O novo candidato só melhora se reduzir claramente essas distâncias e aproximar feições RiverSP dos exutórios.

In [ ]:
if not validacao_anterior.empty and 'distancia_m_feicao_riversp' in validacao_anterior.columns:
    previous_min_km = validacao_anterior['distancia_m_feicao_riversp'].min() / 1000
    previous_max_km = validacao_anterior['distancia_m_feicao_riversp'].max() / 1000
    current_min_km = validation['distancia_m_feicao_riversp'].dropna().min() / 1000
    current_max_km = validation['distancia_m_feicao_riversp'].dropna().max() / 1000
    current_median_km = validation['distancia_m_feicao_riversp'].dropna().median() / 1000
    improves = current_median_km < previous_min_km
    comparison = {
        'candidato_anterior': 'RIVERSP cycle 055 pass 255 tile SA',
        'distancia_anterior_min_km': round(previous_min_km, 2),
        'distancia_anterior_max_km': round(previous_max_km, 2),
        'candidato_atual': 'RIVERSP cycle 055 pass 227 tile SA',
        'distancia_atual_min_km': round(current_min_km, 2),
        'distancia_atual_mediana_km': round(current_median_km, 2),
        'distancia_atual_max_km': round(current_max_km, 2),
        'melhora_em_relacao_ao_anterior': 'sim' if improves else 'nao',
    }
else:
    comparison = {
        'candidato_anterior': 'RIVERSP cycle 055 pass 255 tile SA',
        'distancia_anterior_aproximada_km': '161.7 a 163.7',
        'candidato_atual': 'RIVERSP cycle 055 pass 227 tile SA',
        'observacao': 'Tabela anterior nao encontrada; comparacao numerica automatica nao executada.',
    }

logging.info('Comparacao com RiverSP rejeitado: %s', json.dumps(comparison, ensure_ascii=False))
print(json.dumps(comparison, ensure_ascii=False, indent=2))


## Visualização

A figura mostra os 13 exutórios, as feições RiverSP carregadas e linhas até a feição mais próxima, quando a geometria permite essa indicação.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 9))
try:
    riversp_plot = riversp_gdf.to_crs('EPSG:4326')
    points_plot = points_gdf.to_crs('EPSG:4326')
    lines_plot = gpd.GeoDataFrame(nearest_lines, crs=METRIC_CRS).to_crs('EPSG:4326') if nearest_lines else gpd.GeoDataFrame(geometry=[], crs='EPSG:4326')

    minx, miny, maxx, maxy = points_plot.total_bounds
    pad_x = max((maxx - minx) * 4, 0.04)
    pad_y = max((maxy - miny) * 4, 0.04)

    riversp_plot.plot(ax=ax, color='#2b8cbe', linewidth=1.1, alpha=0.85, label='RiverSP top 1')
    if not lines_plot.empty:
        lines_plot.plot(ax=ax, color='#fb6a4a', linewidth=0.8, alpha=0.65, label='Ligacao ate feicao mais proxima')
    points_plot.plot(ax=ax, color='black', markersize=48, label='Exutorios', zorder=4)
    for _, row in points_plot.iterrows():
        ax.annotate(row['id'], (row.geometry.x, row.geometry.y), xytext=(4, 4), textcoords='offset points', fontsize=8)

    ax.set_xlim(minx - pad_x, maxx + pad_x)
    ax.set_ylim(miny - pad_y, maxy + pad_y)
    ax.set_title('Validacao RiverSP top 1 do ranking - distancias reais')
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.legend(loc='best')
    ax.grid(True, alpha=0.25)
    fig.tight_layout()
    fig.savefig(OUTPUT_FIGURE, dpi=180)
    logging.info('Figura salva: %s', OUTPUT_FIGURE)
    print('OK figura salva:', OUTPUT_FIGURE)
    plt.show()
except Exception as exc:
    logging.exception('Falha ao gerar figura RiverSP top 1')
    raise RuntimeError('FALHA: nao foi possivel gerar a figura de validacao RiverSP top 1.') from exc


## Interpretação da tabela final

- `suporte_riversp = sim`: existe feição RiverSP a até 500 m do exutório.
- `suporte_riversp = nao`: a feição RiverSP mais próxima está longe demais para sustentar suporte preliminar.
- `suporte_riversp = indeterminado`: a leitura ou os atributos não permitem conclusão conservadora.

A decisão final ainda deve considerar qualidade hidrológica, consistência temporal e, se necessário, inspeção PIXC controlada.

## Recomendação do próximo passo

Se o novo RiverSP reduzir as distâncias para a escala local dos exutórios, use-o como candidato principal para análise temporal leve. Se permanecer distante, teste o próximo RiverSP leve do ranking ou um LakeSP relevante antes de baixar PIXC. Só avance para PIXC quando houver uma hipótese espacial bem delimitada.